# 05. Feature Engineering — Stage 0

The objective of this notebook is to construct the feature set for the
Stage-0 volatility-prediction problem.

In `03_eda.ipynb`, we examined the statistical properties of SPY returns,
including return distributions, volatility clustering, temporal dependence,
and the relationship between current and future volatility.

In `04_tda.ipynb`, we investigated the local geometric and topological
structure of the return series using delay-coordinate embeddings, persistent
homology, and persistence landscapes. We also constructed time-dependent
topological quantities such as persistence statistics and persistence
landscape norms.

Rather than introducing a large collection of conventional technical
indicators, we construct features that have a direct connection to the
statistical and topological properties already observed in the data.

We divide the features into two broad families:

1. **Statistical features**

   These describe the recent behavior of returns and volatility, including
   return magnitudes, rolling volatility, and measures of temporal dependence
   and local distributional structure.

2. **Topological features**

   These describe the local geometric and topological structure of the return
   series obtained through delay-coordinate embeddings and persistent
   homology.

The resulting feature vector has the form

$$
X_t =
\left(
X_t^{\mathrm{stat}},
X_t^{\mathrm{TDA}}
\right).
$$

where $X_t^{\mathrm{stat}}$ contains features derived from the conventional
statistical analysis and $X_t^{\mathrm{TDA}}$ contains features derived
from the topological analysis.

The target is based on the relative change between future five-day realized
volatility and current 20-day volatility:

$$
\log\_vol\_ratio_t
=
\log
\left(
\frac{RV_{t,t+5}}
{\sigma_{t,20}}
\right).
$$

For the Stage-0 classification problem, the target will be defined as

$$
y_t =
\begin{cases}
1, & \text{if } \log\_vol\_ratio_t > \theta,\\
0, & \text{otherwise},
\end{cases}
$$

with the threshold $\theta$ chosen in the exploratory analysis.

The purpose of this notebook is therefore to construct a compact,
interpretable, and reproducible feature set that captures both conventional
volatility dynamics and the additional geometric/topological information
identified in the preceding analyses.
The resulting dataset will be used by the Stage-0 baseline, tree-based, and
neural-network models.

A central requirement is that every predictive feature at time $t$ must be
computable using information available on or before $t$.
In particular, the forward five-day realized volatility and the logarithmic
volatility ratio are target-side quantities. They will not be used as
predictive features.
This distinction is essential for avoiding look-ahead bias.

The feature set is deliberately kept relatively compact. The goal is not to
maximize the number of features, but to construct features that encode
distinct and interpretable aspects of the market dynamics identified in the
earlier exploratory analyses.
A later stage of the project will investigate whether additional
data-augmentation strategies based on symmetry, topology, and the statistical
distribution of financial data can improve learning in the relatively
low-to-moderate data regime considered here.

## 1. Project Path and Imports

### 1.1: Define Project Path

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
CLEAN_DATA_FILE = PROCESSED_DATA_DIR / "spy_daily_clean.csv"

### 1.2: Import Libraries

In [2]:
# Install the core TDA packages
#-- uncomment the code below to install risper and persim --#
#%pip install ripser persim

In [3]:
# General scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TDA

from ripser import ripser
from persim import plot_diagrams

# Scaling

from sklearn.preprocessing import StandardScaler

# Project modules
from src.tda import (
    delay_embedding,
    sliding_windows,
    sliding_window_dates,
    persistence_statistics,
    compute_persistence_diagram, 
)

In [4]:
# Verify the TDA packages

import ripser as ripser_module
import persim

from ripser import ripser
from persim import plot_diagrams

print(f"ripser version: {ripser_module.__version__}")
print(f"persim version: {persim.__version__}")

ripser version: 0.6.15
persim version: 0.3.8


## 2. Load the Cleaned Data

The feature engineering is performed using the cleaned SPY dataset produced
in `02_data_cleaning.ipynb`.

Using the cleaned dataset ensures that the feature-engineering stage operates
on the same observations that were validated in the data-cleaning notebook.

In [5]:
spy_data = pd.read_csv(
    CLEAN_DATA_FILE,
    index_col=0,
    parse_dates=True
)

spy_data.head()

,Close,High,Low,Open,Volume
2010-01-04,84.578491,84.623268,83.220220,83.862041,118944600
2010-01-05,84.802376,84.839693,84.220261,84.526247,111579900
2010-01-06,84.862091,85.071055,84.653127,84.720292,116074400
2010-01-07,85.220306,85.324788,84.466539,84.705356,131091100
2010-01-08,85.503906,85.541224,84.824774,84.996420,126402800


In [6]:
print("Shape:", spy_data.shape)
print("First date:", spy_data.index.min())
print("Last date:", spy_data.index.max())

Shape: (4191, 5)
First date: 2010-01-04 00:00:00
Last date: 2026-09-01 00:00:00


## 3. Return-Based Features

The daily log return is the fundamental time series used throughout the
project:

$$
r_t = \log\left(\frac{P_t}{P_{t-1}}\right).
$$

The exploratory analysis showed that return magnitudes contain substantially
more temporal dependence than the raw returns themselves.

We therefore include three basic return-related quantities:

$$
r_t,\qquad |r_t|,\qquad r_t^2.
$$

The raw return preserves the direction of the market movement, while the
absolute and squared returns describe the magnitude of market activity.

The squared return is particularly relevant because realized volatility is
constructed from averages of squared returns.

In [7]:
# Create a working copy for feature engineering.
feature_data = spy_data.copy()

# Daily log return.
feature_data["log_return"] = np.log(
    feature_data["Close"] / feature_data["Close"].shift(1)
)

# Return-based features.
feature_data["return"] = feature_data["log_return"]
feature_data["abs_return"] = feature_data["log_return"].abs()
feature_data["squared_return"] = feature_data["log_return"].pow(2)

feature_data[
    ["return", "abs_return", "squared_return"]
].head()

,return,abs_return,squared_return
2010-01-04,NaN,NaN,NaN
2010-01-05,0.002644,0.002644,6.988439e-06
2010-01-06,0.000704,0.000704,4.955061e-07
2010-01-07,0.004212,0.004212,1.774316e-05
2010-01-08,0.003322,0.003322,1.103780e-05


In [8]:
print(
    feature_data[
        ["return", "abs_return", "squared_return"]
    ].describe()
)

            return   abs_return  squared_return
count  4190.000000  4190.000000     4190.000000
mean      0.000525     0.007171        0.000116
std       0.010777     0.008061        0.000439
min      -0.115887     0.000000        0.000000
25%      -0.003711     0.002008        0.000004
50%       0.000704     0.004909        0.000024
75%       0.005774     0.009654        0.000093
max       0.099863     0.115887        0.013430


A few things are worth noting before we move on:

- We have 4,190 valid returns from the 4,191 price observations, as expected.
- The first return is NaN because there is no preceding price.
- abs_return is exactly \(|r_t|\).
- squared_return is exactly \(r_t^2\).
- The summary statistics agree with what we saw in 03_eda.ipynb.
  
We keep the return column for now even though it duplicates log_return. It makes the feature matrix conceptually explicit; later, when we assemble the final model matrix, we can decide whether to retain both or only one.

## 4. Rolling Volatility Features

The exploratory analysis showed substantial temporal dependence in volatility,
including significant autocorrelation in both absolute and squared returns.

We therefore construct rolling measures of realized volatility at several
time scales.

For a window of length $w$, define

$$
\sigma_{t,w}
=
\sqrt{
\frac{1}{w}
\sum_{j=0}^{w-1} r_{t-j}^2
}.
$$

We use four horizons:

$$
w\in\{5,10,20,60\}.
$$

These correspond approximately to one trading week, two trading weeks, one
trading month, and three trading months.

The 20-day volatility is particularly important because it is the
current-volatility measure used in the definition of the prediction target.

The multiple horizons also provide a simple multiscale description of recent
market activity.

In [9]:
# Rolling volatility features.

VOLATILITY_WINDOWS = [5, 10, 20, 60]

for window in VOLATILITY_WINDOWS:
    feature_data[f"vol_{window}d"] = np.sqrt(
        feature_data["squared_return"]
        .rolling(window)
        .mean()
    )

feature_data[
    [
        "vol_5d",
        "vol_10d",
        "vol_20d",
        "vol_60d"
    ]
].tail()

,vol_5d,vol_10d,vol_20d,vol_60d
2026-08-26,0.004620,0.004814,0.007548,0.008729
2026-08-27,0.003955,0.004755,0.006729,0.008768
2026-08-28,0.003653,0.004768,0.006554,0.008726
2026-08-31,0.003661,0.004624,0.005780,0.008720
2026-09-01,0.004569,0.004641,0.004452,0.008090


In [10]:
# inspect them
feature_data[
    [
        "vol_5d",
        "vol_10d",
        "vol_20d",
        "vol_60d"
    ]
].describe()

,vol_5d,vol_10d,vol_20d,vol_60d
count,4186.000000,4181.000000,4171.000000,4131.000000
mean,0.008696,0.008976,0.009213,0.009601
std,0.006394,0.006004,0.005642,0.004980
min,0.000828,0.001213,0.002064,0.003215
25%,0.004741,0.005404,0.005888,0.006861
50%,0.007307,0.007507,0.007836,0.008114
75%,0.010789,0.010848,0.010850,0.010681
max,0.084543,0.070240,0.058156,0.038648


A useful **observation** from the summary is that the maximum volatility decreases as the window gets longer:
$$
0.0845 \rightarrow 0.0702 \rightarrow 0.0582 \rightarrow 0.0386.
$$
That's natural: short windows respond much more strongly to individual extreme returns, while longer windows smooth them.

### 4.1 Relative Volatility Features

The absolute level of volatility is only one aspect of the current market
regime. We also want to describe volatility relative to a longer or shorter
time scale.

We therefore construct ratios between rolling volatility measures:

$$
R^{(5/20)}_t
=
\frac{\sigma_{t,5}}{\sigma_{t,20}},
$$

and

$$
R^{(20/60)}_t
=
\frac{\sigma_{t,20}}{\sigma_{t,60}}.
$$

The first ratio measures whether very recent volatility is elevated relative
to the 20-day volatility level. The second compares the current monthly
volatility environment with a longer-term 60-day baseline.

These quantities are computed entirely from information available at time
$t$, so they do not introduce look-ahead bias.

In [11]:
feature_data["vol_ratio_5_20"] = (
    feature_data["vol_5d"] / feature_data["vol_20d"]
)

feature_data["vol_ratio_20_60"] = (
    feature_data["vol_20d"] / feature_data["vol_60d"]
)

feature_data[
    ["vol_ratio_5_20", "vol_ratio_20_60"]
].describe()

,vol_ratio_5_20,vol_ratio_20_60
count,4171.000000,4131.000000
mean,0.947232,0.966935
std,0.335586,0.255669
min,0.119317,0.314460
25%,0.704744,0.788470
50%,0.936574,0.963434
75%,1.160233,1.138381
max,1.914154,1.704760


## 5. Volatility-Clustering Features

The exploratory analysis showed substantial temporal dependence in return
magnitudes.

In particular, `03_eda.ipynb` examined the autocorrelation of absolute and
squared returns at lags

$$
\{1,5,10,20\}.
$$

Both quantities exhibited persistent autocorrelation across these lags. The
squared returns showed particularly strong dependence at short lags.

We therefore include lagged absolute and squared returns as features:

$$
|r_{t-\ell}|,
\qquad
r_{t-\ell}^2,
\qquad
\ell\in\{1,5,10,20\}.
$$

These features provide the model with explicit information about recent
volatility activity at several temporal separations.

We do not introduce an equally large collection of lagged raw returns because
the exploratory analysis focused on the temporal dependence of return
magnitudes rather than the raw returns themselves.

In [12]:
# Lagged return-magnitude features.

RETURN_LAGS = [1, 5, 10, 20]

for lag in RETURN_LAGS:
    feature_data[f"abs_return_lag_{lag}"] = (
        feature_data["abs_return"].shift(lag)
    )

    feature_data[f"squared_return_lag_{lag}"] = (
        feature_data["squared_return"].shift(lag)
    )

feature_data[
    [
        "abs_return_lag_1",
        "abs_return_lag_5",
        "abs_return_lag_10",
        "abs_return_lag_20",
        "squared_return_lag_1",
        "squared_return_lag_5",
        "squared_return_lag_10",
        "squared_return_lag_20",
    ]
].head(25)

,abs_return_lag_1,abs_return_lag_5,abs_return_lag_10,abs_return_lag_20,squared_return_lag_1,squared_return_lag_5,squared_return_lag_10,squared_return_lag_20
2010-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,0.002644,NaN,NaN,NaN,6.988439e-06,NaN,NaN,NaN
2010-01-07,0.000704,NaN,NaN,NaN,4.955061e-07,NaN,NaN,NaN
2010-01-08,0.004212,NaN,NaN,NaN,1.774316e-05,NaN,NaN,NaN
2010-01-11,0.003322,NaN,NaN,NaN,1.103780e-05,NaN,NaN,NaN
2010-01-12,0.001396,0.002644,NaN,NaN,1.947539e-06,6.988439e-06,NaN,NaN
2010-01-13,0.009370,0.000704,NaN,NaN,8.779575e-05,4.955061e-07,NaN,NaN
2010-01-14,0.008411,0.004212,NaN,NaN,7.074317e-05,1.774316e-05,NaN,NaN
2010-01-15,0.002701,0.003322,NaN,NaN,7.293137e-06,1.103780e-05,NaN,NaN


In [13]:
feature_data[
    [
        "abs_return_lag_1",
        "abs_return_lag_5",
        "abs_return_lag_10",
        "abs_return_lag_20",
        "squared_return_lag_1",
        "squared_return_lag_5",
        "squared_return_lag_10",
        "squared_return_lag_20",
    ]
].isna().sum()

abs_return_lag_1          2
abs_return_lag_5          6
abs_return_lag_10        11
abs_return_lag_20        21
squared_return_lag_1      2
squared_return_lag_5      6
squared_return_lag_10    11
squared_return_lag_20    21
dtype: int64

## 6. Topological Features

The exploratory analysis in `04_tda.ipynb` showed that the local return
dynamics can be represented using delay-coordinate embeddings and persistent
homology.

Rather than recomputing the persistent homology calculations in this
notebook, we use the time-indexed TDA feature table saved by
`04_tda.ipynb`.

The table contains two types of topological summaries.

First, we use persistence statistics from the \(H_0\) and \(H_1\) persistence
diagrams:

$$
P_{\mathrm{total}},
\qquad
P_{\mathrm{mean}},
\qquad
P_{\mathrm{max}},
$$

together with the number of \(H_1\) features.

Second, we use the persistence-landscape norms:

$$
L^1,\qquad L^2,\qquad L^\infty.
$$

These quantities provide different scalar summaries of the local topological
structure of the return dynamics.

At this stage, we retain the complete set of TDA summaries rather than
selecting features based on their correlation with the target. Feature
redundancy and model-specific usefulness will be examined later.

The TDA features are aligned with the statistical features using their
associated dates.

**Load the saved TDA analysis:**

In [14]:
TDA_FEATURE_FILE = PROCESSED_DATA_DIR / "spy_tda_features.csv"

tda_features = pd.read_csv(
    TDA_FEATURE_FILE,
    parse_dates=["date"]
)

tda_features.head()

,date,h0_total_persistence,h0_mean_persistence,h0_max_persistence,h1_total_persistence,h1_mean_persistence,h1_max_persistence,h1_n_features,h1_l1_norm,h1_l2_norm,h1_sup_norm
0,2010-02-02,0.110153,0.006120,0.013418,0.000000,0.000000,0.000000,0,0.000000e+00,0.00000,0.00000
1,2010-02-03,0.113985,0.006333,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2,2010-02-04,0.126696,0.007039,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
3,2010-02-05,0.134904,0.007495,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
4,2010-02-08,0.136133,0.007563,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084


In [15]:
# inspect

print("Shape:", tda_features.shape)
print("Columns:")
print(tda_features.columns.tolist())

Shape: (4171, 11)
Columns:
['date', 'h0_total_persistence', 'h0_mean_persistence', 'h0_max_persistence', 'h1_total_persistence', 'h1_mean_persistence', 'h1_max_persistence', 'h1_n_features', 'h1_l1_norm', 'h1_l2_norm', 'h1_sup_norm']


### 6.1 Aligning TDA and Statistical Features

The statistical features and TDA features are indexed differently.

The statistical features begin with the first available return, whereas the
TDA features begin only after the first complete sliding window.

We therefore align the two datasets using the observation date.

An inner merge is appropriate because a Stage-0 observation should contain
both statistical and topological information. Observations for which the TDA
features cannot be computed are therefore excluded from the final feature
matrix.

In [16]:
feature_data = feature_data.merge(
    tda_features,
    left_index=True,
    right_on="date",
    how="inner"
)

feature_data = feature_data.set_index("date")

feature_data.head()

,Close,High,Low,Open,Volume,log_return,return,abs_return,squared_return,vol_5d,...,h0_total_persistence,h0_mean_persistence,h0_max_persistence,h1_total_persistence,h1_mean_persistence,h1_max_persistence,h1_n_features,h1_l1_norm,h1_l2_norm,h1_sup_norm
date,,,,,,,,,,,,,,,,,,,,,
2010-02-02,82.376892,82.533615,81.257438,81.541037,216327900,0.012031,0.012031,0.012031,0.000145,0.011471,...,0.110153,0.006120,0.013418,0.000000,0.000000,0.000000,0,0.000000e+00,0.00000,0.00000
2010-02-03,81.966423,82.451521,81.727606,82.003735,172730700,-0.004995,-0.004995,0.004995,0.000025,0.011493,...,0.113985,0.006333,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2010-02-04,79.436470,81.369392,79.421541,81.332080,356715700,-0.031352,-0.031352,0.031352,0.000983,0.017379,...,0.126696,0.007039,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2010-02-05,79.600632,79.764813,78.048321,79.525997,493585800,0.002064,0.002064,0.002064,0.000004,0.016704,...,0.134904,0.007495,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2010-02-08,79.025993,80.100671,78.966288,79.660350,224166900,-0.007245,-0.007245,0.007245,0.000052,0.015553,...,0.136133,0.007563,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084


In [17]:
# check

print("Shape:", feature_data.shape)
print("First date:", feature_data.index.min())
print("Last date:", feature_data.index.max())

Shape: (4171, 33)
First date: 2010-02-02 00:00:00
Last date: 2026-09-01 00:00:00


In [18]:
# inspect the complete feature list

print(feature_data.columns.tolist())

['Close', 'High', 'Low', 'Open', 'Volume', 'log_return', 'return', 'abs_return', 'squared_return', 'vol_5d', 'vol_10d', 'vol_20d', 'vol_60d', 'vol_ratio_5_20', 'vol_ratio_20_60', 'abs_return_lag_1', 'squared_return_lag_1', 'abs_return_lag_5', 'squared_return_lag_5', 'abs_return_lag_10', 'squared_return_lag_10', 'abs_return_lag_20', 'squared_return_lag_20', 'h0_total_persistence', 'h0_mean_persistence', 'h0_max_persistence', 'h1_total_persistence', 'h1_mean_persistence', 'h1_max_persistence', 'h1_n_features', 'h1_l1_norm', 'h1_l2_norm', 'h1_sup_norm']


## 7. Define the Stage-0 Feature Set

The merged dataset contains the original market variables together with the
statistical and topological quantities constructed above.

For the Stage-0 prediction models, we retain features that were motivated by
the exploratory analysis and TDA analysis.

The raw OHLCV variables are not included in the initial predictive feature
matrix. Although these variables are observable, they were not the focus of
the feature-engineering analysis developed in the preceding notebooks.

We also remove the duplicate `return` column because it is identical to
`log_return`.

The resulting feature set consists of:

1. return-based features,
2. rolling volatility features,
3. relative-volatility features,
4. lagged return-magnitude features, and
5. topological features.

This gives a total of 27 initial predictive features.

The feature set is intentionally kept fixed at this stage. Feature selection
and model-specific regularization will be considered during the modeling
stage rather than using the target to select features here.

In [19]:
FEATURE_COLUMNS = [
    # Return features
    "log_return",
    "abs_return",
    "squared_return",

    # Rolling volatility
    "vol_5d",
    "vol_10d",
    "vol_20d",
    "vol_60d",

    # Relative volatility
    "vol_ratio_5_20",
    "vol_ratio_20_60",

    # Lagged return magnitude
    "abs_return_lag_1",
    "abs_return_lag_5",
    "abs_return_lag_10",
    "abs_return_lag_20",
    "squared_return_lag_1",
    "squared_return_lag_5",
    "squared_return_lag_10",
    "squared_return_lag_20",

    # TDA features
    "h0_total_persistence",
    "h0_mean_persistence",
    "h0_max_persistence",
    "h1_total_persistence",
    "h1_mean_persistence",
    "h1_max_persistence",
    "h1_n_features",
    "h1_l1_norm",
    "h1_l2_norm",
    "h1_sup_norm",
]

In [20]:
print("Number of features:", len(FEATURE_COLUMNS))

Number of features: 27


In [21]:
stage0_features = feature_data[FEATURE_COLUMNS].copy()

print("Shape:", stage0_features.shape)
stage0_features.head()

Shape: (4171, 27)


,log_return,abs_return,squared_return,vol_5d,vol_10d,vol_20d,vol_60d,vol_ratio_5_20,vol_ratio_20_60,abs_return_lag_1,...,h0_total_persistence,h0_mean_persistence,h0_max_persistence,h1_total_persistence,h1_mean_persistence,h1_max_persistence,h1_n_features,h1_l1_norm,h1_l2_norm,h1_sup_norm
date,,,,,,,,,,,,,,,,,,,,,
2010-02-02,0.012031,0.012031,0.000145,0.011471,0.013006,0.010433,NaN,1.099483,NaN,0.015431,...,0.110153,0.006120,0.013418,0.000000,0.000000,0.000000,0,0.000000e+00,0.00000,0.00000
2010-02-03,-0.004995,0.004995,0.000025,0.011493,0.012696,0.010476,NaN,1.096994,NaN,0.012031,...,0.113985,0.006333,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2010-02-04,-0.031352,0.031352,0.000983,0.017379,0.014893,0.012605,NaN,1.378803,NaN,0.004995,...,0.126696,0.007039,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2010-02-05,0.002064,0.002064,0.000004,0.016704,0.013092,0.012578,NaN,1.328006,NaN,0.031352,...,0.134904,0.007495,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084
2010-02-08,-0.007245,0.007245,0.000052,0.015553,0.013192,0.012660,NaN,1.228481,NaN,0.002064,...,0.136133,0.007563,0.013418,0.001681,0.001681,0.001681,1,7.061733e-07,0.00002,0.00084


## 8. Construct the Prediction Target

The prediction problem is based on the relative change between future five-day
realized volatility and current 20-day volatility.

For each date \(t\), define the future five-day realized volatility as

$$
RV_{t,t+5}
=
\sqrt{
\frac{1}{5}
\sum_{k=1}^{5} r_{t+k}^2
}.
$$

The corresponding relative-volatility quantity is

$$
\mathrm{log\_vol\_ratio}_t
=
\log
\left(
\frac{RV_{t,t+5}}
{\sigma_{t,20}}
\right).
$$

For the Stage-0 classification problem, we use the threshold

$$
\theta = 0.2.
$$

The binary target is therefore

$$
y_t =
\begin{cases}
1, & \mathrm{log\_vol\_ratio}_t > 0.2,\\
0, & \mathrm{log\_vol\_ratio}_t \leq 0.2.
\end{cases}
$$

Thus, \(y_t=1\) indicates that future five-day realized volatility is
approximately \(22\%\) or more above the current 20-day volatility, since

$$
e^{0.2}\approx 1.22.
$$

The future realized volatility and logarithmic volatility ratio are target-side
quantities. They must not be included as predictive features, since doing so
would introduce look-ahead bias.

In [22]:
# construct the future realized volatility
TARGET_THRESHOLD = 0.2
TARGET_WINDOW = 5

future_squared_returns = (
    feature_data["squared_return"]
    .shift(-1)
)

future_5d_rv = np.sqrt(
    future_squared_returns
    .rolling(TARGET_WINDOW)
    .mean()
    .shift(-(TARGET_WINDOW - 1))
)

feature_data["future_5d_rv"] = future_5d_rv

In [23]:
# construct the log ratio
feature_data["log_vol_ratio"] = np.log(
    feature_data["future_5d_rv"]
    / feature_data["vol_20d"]
)

In [24]:
# construct the binary target
feature_data["target"] = (
    feature_data["log_vol_ratio"] > TARGET_THRESHOLD
).astype(int)

Check the target

In [25]:
feature_data[
    [
        "vol_20d",
        "future_5d_rv",
        "log_vol_ratio",
        "target",
    ]
].head(10)

,vol_20d,future_5d_rv,log_vol_ratio,target
date,,,,
2010-02-02,0.010433,0.015624,0.403763,1
2010-02-03,0.010476,0.015488,0.390931,1
2010-02-04,0.012605,0.008060,-0.447158,0
2010-02-05,0.012578,0.008016,-0.450557,0
2010-02-08,0.012660,0.010124,-0.223520,0
2010-02-09,0.012960,0.008707,-0.397793,0
2010-02-10,0.012797,0.009052,-0.346196,0
2010-02-11,0.012871,0.007818,-0.498495,0
2010-02-12,0.012858,0.007810,-0.498580,0


In [26]:
print(
    feature_data["target"].value_counts()
)

print(
    feature_data["target"].value_counts(normalize=True)
)

target
0    3156
1    1015
Name: count, dtype: int64
target
0    0.756653
1    0.243347
Name: proportion, dtype: float64


In [27]:
# verify the target construction against the exploratory analysis
print(
    feature_data["log_vol_ratio"].describe()
)

print(
    "Target threshold:",
    TARGET_THRESHOLD
)

count    4166.000000
mean       -0.125770
std         0.520109
min        -2.251823
25%        -0.458419
50%        -0.110625
75%         0.189944
max         1.700821
Name: log_vol_ratio, dtype: float64
Target threshold: 0.2


## 9. Final Feature-Target Alignment

The feature construction process introduces missing values at the beginning
of the sample because some features require historical observations.

The prediction target introduces missing values at the end of the sample
because future five-day realized volatility cannot be computed for the final
five observations.

We now retain only observations for which all 27 predictive features and the
target are available.

No imputation is performed. The missing values arise from the mathematical
construction of lagged, rolling, and future quantities rather than from
missing observations in the original cleaned market data.

### 9.1: create the final Stage-0 dataset

In [31]:
stage0_data = feature_data[
    FEATURE_COLUMNS + ["target"]
].dropna()

### 9.2: Inspect

In [32]:
print("Shape:", stage0_data.shape)
print("First date:", stage0_data.index.min())
print("Last date:", stage0_data.index.max())

Shape: (4131, 28)
First date: 2010-03-31 00:00:00
Last date: 2026-09-01 00:00:00


In [33]:
print("Missing values:")
print(stage0_data.isna().sum().sum())

Missing values:
0


In [34]:
print("Target distribution:")
print(stage0_data["target"].value_counts())

print("\nTarget proportions:")
print(stage0_data["target"].value_counts(normalize=True))

Target distribution:
target
0    3118
1    1013
Name: count, dtype: int64

Target proportions:
target
0    0.754781
1    0.245219
Name: proportion, dtype: float64


Let's also verify that none of the target-side quantities accidentally entered our 27-feature matrix:

In [35]:
TARGET_COLUMNS = [
    "future_5d_rv",
    "log_vol_ratio",
    "target",
]

for column in TARGET_COLUMNS:
    print(
        f"{column}:",
        column in FEATURE_COLUMNS
    )

future_5d_rv: False
log_vol_ratio: False
target: False


## 10. Save the Stage-0 Dataset

The final Stage-0 dataset contains the 27 predictive features and the binary
prediction target.

The date index is retained so that chronological train/validation/test splits
can be constructed in the subsequent modeling notebooks.

The resulting dataset contains no missing values and is saved in the
processed-data directory.

In [36]:
STAGE0_DATA_FILE = (
    PROCESSED_DATA_DIR / "spy_stage0_features.csv"
)

stage0_data.to_csv(
    STAGE0_DATA_FILE,
    index=True,
)

print(f"Saved Stage-0 dataset to: {STAGE0_DATA_FILE}")

Saved Stage-0 dataset to: /Users/amitkumar/Desktop/Computing/Projects/Finance/market-volatility/data/processed/spy_stage0_features.csv


### 10.1 Verify the Saved Dataset

As a final reproducibility check, reload the saved Stage-0 dataset from disk
and verify its shape, date range, and absence of missing values.

In [37]:
# Reload the saved Stage-0 dataset.

reloaded_stage0 = pd.read_csv(
    STAGE0_DATA_FILE,
    index_col=0,
    parse_dates=True
)

print("Shape:", reloaded_stage0.shape)
print("First date:", reloaded_stage0.index.min())
print("Last date:", reloaded_stage0.index.max())
print("Missing values:", reloaded_stage0.isna().sum().sum())

Shape: (4131, 28)
First date: 2010-03-31 00:00:00
Last date: 2026-09-01 00:00:00
Missing values: 0


In [38]:
print("Feature columns:", len(FEATURE_COLUMNS))
print("Target column:", "target" in reloaded_stage0.columns)

Feature columns: 27
Target column: True
